# 7-Zone 28-Stem Cavern Stage — Google Colab Pro 1440p60 & 1080p60 Master Render Pipeline
**Lady Gaga "Abracadabra" 28-Stem Spatial 3D Cavern Visualiser**

This notebook runs the deterministic headless rendering pipeline powered by Three.js, PCFSoftShadowMap (4096), MeshReflectorMaterial planar ray tracing, AgX filmic tone mapping, and Nvidia NVENC hardware encoding.

### Deliverables Produced:
1. `Abracadabra_Stage_1440p60.mp4` (Native 2560x1440 60 FPS Master)
2. `Abracadabra_Stage_1080p60.mp4` (Supersampled 1920x1080 60 FPS Master)

Both formats are rendered and encoded in a single high-efficiency pass with full audio sync, and automatically saved directly into your Google Drive (`/content/drive/MyDrive/`).

> **Important**: Ensure your Colab runtime is set to GPU: **Runtime -> Change runtime type -> T4 GPU (or higher)**.

In [ ]:
# 1. Mount Google Drive to persist rendered 1440p and 1080p video masters
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Check assigned Nvidia GPU hardware (T4 / A100 / V100)
!nvidia-smi

In [ ]:
# 3. Install system dependencies (FFmpeg with NVENC support, Xvfb virtual display, Node.js 20 LTS)
!apt-get update -qq
!apt-get install -y -qq ffmpeg xvfb libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs
!node -v && npm -v

In [ ]:
# 4. Clone or prepare the visualiser repository
import os
%cd /content
if not os.path.exists('/content/visualiser'):
    !git clone https://github.com/okaythi/visualiser.git /content/visualiser || mkdir -p /content/visualiser
%cd /content/visualiser
!npm install
!npx puppeteer browsers install chrome

In [ ]:
# 5. Build production bundle & start local preview server
import subprocess
import time
import urllib.request

print("Building production client bundle...")
subprocess.run(["npm", "run", "build"], check=True)

print("Starting Vite preview server in background...")
server_proc = subprocess.Popen(["npm", "run", "preview", "--", "--port", "5173", "--host"])

# Wait for server to become responsive
server_ready = False
for i in range(30):
    try:
        with urllib.request.urlopen("http://localhost:5173/?render=1") as resp:
            if resp.status == 200:
                print("✓ Vite server ready at http://localhost:5173")
                server_ready = True
                break
    except Exception:
        time.sleep(1)

if not server_ready:
    print("Warning: preview server check timed out, proceeding with render...")

In [ ]:
# 6. Execute deterministic headless render pipeline (1440p60 & 1080p60 Dual NVENC output)
# Xvfb creates a virtual 2560x1440 24-bit framebuffer with GLX hardware acceleration
!xvfb-run -s "-screen 0 2560x1440x24 -ac +extension GLX +render -noreset" \
  node scripts/render_headless.mjs --format both --nvenc

In [ ]:
# 7. Verify outputs and save directly to Google Drive
import os
import shutil

drive_dir = "/content/drive/MyDrive"
files = [
    "Abracadabra_Stage_1440p60.mp4",
    "Abracadabra_Stage_1080p60.mp4"
]

print("=== Output Verification & Google Drive Transfer ===")
for filename in files:
    if os.path.exists(filename):
        size_mb = os.path.getsize(filename) / (1024 * 1024)
        dest_path = os.path.join(drive_dir, filename)
        shutil.copy2(filename, dest_path)
        print(f"✓ Copied {filename} ({size_mb:.2f} MB) -> {dest_path}")
    else:
        print(f"⚠ Warning: {filename} was not found in current directory.")

print("\nRender master delivery complete! Check your Google Drive root folder.")